# Bronze Layer — Managed Delta Tables with Auto Loader Streaming Ingestion

## Design Decisions
- **Managed Delta tables** (not external) so Unity Catalog owns both metadata and data lifecycle.
- **Auto Loader (`cloudFiles`)** for file-based incremental streaming ingestion — only new, unprocessed files are read on each run via a checkpoint; safe to re-run at any frequency.
- **`trigger(availableNow=True)`** makes the stream behave like a scheduled batch — it processes all files that arrived since the last checkpoint, then stops. Ideal for orchestrated pipeline jobs.
- **`cloudFiles.schemaLocation`** persists the inferred schema to ADLS so it is not re-inferred on every run, and schema evolution is handled automatically.
- **`_metadata.file_path`** captures the actual source path at read-time (Auto Loader built-in metadata column).
- **NOT NULL constraints** applied at the Bronze layer as the first data quality gate.
- **Partitioned by `ingestion_date`** for query pruning and efficient time-travel.

> Prerequisite: Run `0.config.ipynb` first (or let the Workflow chain handle it via `%run`).
> Source files should land in dedicated subdirectories: `{BRONZE_PATH}drivers/` and `{BRONZE_PATH}results/`.


In [0]:
%run ./0.config

In [0]:
# ── Step 1: Ensure catalog / schema exist ────────────────────────────────────
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA} MANAGED LOCATION '{BRONZE_PATH}'")

In [0]:
# ── Step 2: Create target managed Delta tables (DDL with NOT NULL constraints) ─
# BIGINT used for all integer fields — Spark's JSON reader infers all integers
# as Long (64-bit) by default. Declaring INT (32-bit) causes a type conflict
# when mergeSchema=true tries to reconcile INT vs LONG at write time.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq(BRONZE_SCHEMA, 'drivers')} (
    driverId       BIGINT    NOT NULL,
    driverRef      STRING    NOT NULL,
    number         BIGINT,
    code           STRING,
    name           STRUCT<forename: STRING, surname: STRING>,
    dob            STRING,
    nationality    STRING,
    url            STRING,
    ingestion_date DATE      NOT NULL,
    source_file    STRING
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality'                    = 'bronze'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq(BRONZE_SCHEMA, 'results')} (
    resultId        BIGINT  NOT NULL,
    raceId          BIGINT  NOT NULL,
    driverId        BIGINT  NOT NULL,
    constructorId   BIGINT  NOT NULL,
    number          BIGINT,
    grid            BIGINT,
    position        BIGINT,
    positionText    STRING,
    positionOrder   BIGINT,
    points          DOUBLE,
    laps            BIGINT,
    time            STRING,
    milliseconds    BIGINT,
    fastestLap      BIGINT,
    rank            BIGINT,
    fastestLapTime  STRING,
    fastestLapSpeed STRING,
    statusId        BIGINT,
    ingestion_date  DATE    NOT NULL,
    source_file     STRING
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality'                    = 'bronze'
)
""")

In [0]:
# ── Step 3: Auto Loader — Ingest bronze.drivers (incremental, checkpoint-backed) ─
# cloudFiles monitors {BRONZE_PATH}drivers/ and processes only files that are
# new since the last checkpoint. trigger(availableNow=True) drains all pending
# files then stops — behaves like a batch but is fully incremental.

from pyspark.sql import functions as F

drivers_query = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",          "json")
    .option("cloudFiles.inferColumnTypes","true")
    .option("cloudFiles.schemaLocation",  f"{BRONZE_PATH}_schema/drivers")
    .option("multiLine",                  "true")
    .load(f"{BRONZE_PATH}drivers/")
    .withColumn("ingestion_date", F.current_date())
    .withColumn("source_file",    F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{BRONZE_PATH}_checkpoint/drivers")
    .trigger(availableNow=True)
    .toTable(fq(BRONZE_SCHEMA, "drivers"))
)
drivers_query.awaitTermination()
print(f"drivers loaded: {spark.table(fq(BRONZE_SCHEMA, 'drivers')).count():,} rows")


In [0]:
# ── Step 4: Auto Loader — Ingest bronze.results (incremental, checkpoint-backed) ─

results_query = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",          "json")
    .option("cloudFiles.inferColumnTypes","true")
    .option("cloudFiles.schemaLocation",  f"{BRONZE_PATH}_schema/results")
    .option("multiLine",                  "true")
    .load(f"{BRONZE_PATH}results/")
    .withColumn("ingestion_date", F.current_date())
    .withColumn("source_file",    F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{BRONZE_PATH}_checkpoint/results")
    .trigger(availableNow=True)
    .toTable(fq(BRONZE_SCHEMA, "results"))
)
results_query.awaitTermination()
print(f"results loaded: {spark.table(fq(BRONZE_SCHEMA, 'results')).count():,} rows")


In [0]:
# ── Step 5: Data Quality Validation — row counts & NOT NULL checks ────────────

def validate_bronze(schema: str, table: str, pk_col: str):
    full_name = fq(schema, table)
    df = spark.table(full_name)
    total      = df.count()
    null_pk    = df.filter(F.col(pk_col).isNull()).count()
    null_date  = df.filter(F.col("ingestion_date").isNull()).count()

    assert total   > 0,  f"[DQ FAIL] {full_name}: table is empty!"
    assert null_pk == 0, f"[DQ FAIL] {full_name}: {null_pk} NULL values in primary key '{pk_col}'"
    assert null_date == 0, f"[DQ FAIL] {full_name}: {null_date} NULL values in ingestion_date"

    print(f"[DQ PASS] {full_name}: {total:,} rows | 0 NULL PKs | 0 NULL dates")

validate_bronze(BRONZE_SCHEMA, "drivers", "driverId")
validate_bronze(BRONZE_SCHEMA, "results", "resultId")

In [ ]:
# ── Step 6: Maintenance — OPTIMIZE ───────────────────────────────────────────
for tbl in ["drivers", "results"]:
    spark.sql(f"OPTIMIZE {fq(BRONZE_SCHEMA, tbl)}")
    print(f"OPTIMIZE complete: {fq(BRONZE_SCHEMA, tbl)}")

# Schedule VACUUM on a maintenance job — not inline with every pipeline run.
# for tbl in ["drivers", "results"]:
#     spark.sql(f"VACUUM {fq(BRONZE_SCHEMA, tbl)} RETAIN 168 HOURS")


In [ ]:
# ── Step 7: Unity Catalog — Table & Column Comments ──────────────────────────
# Comments power the UC Data Explorer and make tables self-documenting.

spark.sql(f"""COMMENT ON TABLE {fq(BRONZE_SCHEMA, 'drivers')}
IS 'Raw Formula 1 driver records ingested from the ADLS Bronze container.'""")

spark.sql(f"""COMMENT ON TABLE {fq(BRONZE_SCHEMA, 'results')}
IS 'Raw Formula 1 race result records ingested from the ADLS Bronze container.'""")

spark.sql(f"""ALTER TABLE {fq(BRONZE_SCHEMA, 'drivers')}
ALTER COLUMN driverId   COMMENT 'Unique driver identifier (primary key)'""")
spark.sql(f"""ALTER TABLE {fq(BRONZE_SCHEMA, 'drivers')}
ALTER COLUMN driverRef  COMMENT 'Short driver reference slug (e.g. hamilton)'""")
spark.sql(f"""ALTER TABLE {fq(BRONZE_SCHEMA, 'drivers')}
ALTER COLUMN ingestion_date COMMENT 'Date the record was loaded into the Bronze layer'""")

spark.sql(f"""ALTER TABLE {fq(BRONZE_SCHEMA, 'results')}
ALTER COLUMN resultId  COMMENT 'Unique race result identifier (primary key)'""")
spark.sql(f"""ALTER TABLE {fq(BRONZE_SCHEMA, 'results')}
ALTER COLUMN raceId    COMMENT 'Foreign key to the races table'""")
spark.sql(f"""ALTER TABLE {fq(BRONZE_SCHEMA, 'results')}
ALTER COLUMN ingestion_date COMMENT 'Date the record was loaded into the Bronze layer'""")

print("Table and column comments applied.")
